In [10]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

In [5]:
load_dotenv()

True

In [6]:
llm = ChatOpenAI()

In [7]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str
    

In [8]:
def generate_joke(state: JokeState) -> JokeState:
    prompt = f"""
    Generate a joke about the topic: {state["topic"]}
    """
    response = llm.invoke(prompt)
    return {"joke": response.content}



In [9]:
def generate_explanation(state: JokeState) -> JokeState:
    prompt = f"""
    Generate an explanation for the joke: {state["joke"]}
    """
    response = llm.invoke(prompt)
    return {"explanation": response.content}


In [12]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)

checkpoint = InMemorySaver()

workflow = graph.compile(checkpointer=checkpoint)

In [13]:
config1 = {'configurable': {'thread_id': 1}}
workflow.invoke({'topic': 'football'}, config=config1)


{'topic': 'football',
 'joke': 'Why did the football coach go to the bank?\nTo get his quarterback!',
 'explanation': 'This joke plays on the idea of a quarterback being a position in football and also a term used in banking to refer to a customer who is a regular or preferred client of a bank. The punchline "to get his quarterback" suggests that the football coach went to the bank to retrieve or secure his preferred or trusted player, in this case, a quarterback. It\'s a clever play on words that combines both the sports and banking contexts.'}

In [14]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'football', 'joke': 'Why did the football coach go to the bank?\nTo get his quarterback!', 'explanation': 'This joke plays on the idea of a quarterback being a position in football and also a term used in banking to refer to a customer who is a regular or preferred client of a bank. The punchline "to get his quarterback" suggests that the football coach went to the bank to retrieve or secure his preferred or trusted player, in this case, a quarterback. It\'s a clever play on words that combines both the sports and banking contexts.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c1078-a4c2-6d54-8002-b8a3692dbe80'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-05T21:55:55.437581+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c1078-9508-65da-8001-bb3005a07312'}}, tasks=(), interrupts=())

In [15]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'football', 'joke': 'Why did the football coach go to the bank?\nTo get his quarterback!', 'explanation': 'This joke plays on the idea of a quarterback being a position in football and also a term used in banking to refer to a customer who is a regular or preferred client of a bank. The punchline "to get his quarterback" suggests that the football coach went to the bank to retrieve or secure his preferred or trusted player, in this case, a quarterback. It\'s a clever play on words that combines both the sports and banking contexts.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c1078-a4c2-6d54-8002-b8a3692dbe80'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-05T21:55:55.437581+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c1078-9508-65da-8001-bb3005a07312'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'foot

In [20]:
workflow.get_state({'configurable':{'thread_id':'thread-1',"checkpoint_id":'1f1c1078-7776-6788-8000-c52551e2827c'}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_id': '1f1c1078-7776-6788-8000-c52551e2827c'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())